In [ ]:
import sys
sys.path.insert(0, '.')
import figio as fx
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_palette(fx.PALETTE)

# Units and tests
# ---------------
# Box panels plot one value per allele or per withheld molecule - folds averaged
# within a seed, then seeds averaged - so a point is a molecule, never a run.
# Plotting per-fold or per-seed values instead would pseudo-replicate: 15 runs of
# one model on one allele are not 15 independent observations.
#
# Wherever DeepNeo is in the panel the units are collapsed onto the beta chain
# first (fx.by_beta). DeepNeo holds out beta chains where the other models hold
# out alpha/beta pairs, so the two sides share no unit names at all; without the
# collapse there is nothing for a paired test to match on, which is why the old
# code fell back to Mann-Whitney for exactly those pairs and to
# `wilcoxon(a.iloc[:n], b.iloc[:n])` - pairing by row position - for the rest.
#
# Brackets are Holm-corrected paired Wilcoxon across all pairs in the panel.
#
# The published tools
# -------------------
# pipeline.ref_level() scores NetMHCIIpan-4.3 and MixMHC2pred-2.0 as part of every
# analysis, so every scoring table carries them and fx.* drops them unless asked
# (refs=True). Only Per Molecule asks.
#
# They are held to one panel on purpose. Both were trained on IEDB and
# immunopeptidomics data that overlaps this test set, so their numbers are an
# upper bound, not a like-for-like result - which is exactly why the Per Molecule
# result is worth showing: four representations beat them anyway, and a bias that
# runs in the tools' favour cannot explain that away. The panels where the tools
# come out ahead (H2-Out, LOMO-H2) are the ones the overlap could explain, so
# putting them there would be reporting the confound as a finding. Those numbers
# go to the supplementary table instead, with the caveat attached.


In [ ]:
# plot2a: serotype breakdown, Qualitative on top and MS below.
# refs=True in every panel: NetMHCIIpan-4.3 and MixMHC2pred-2.0 are scored on the
# same test set and the same rows as the representations beside them, by
# pipeline.ref_level(). They carry no error whisker because there is no seed to
# vary - both are deterministic published binaries - which is what draw_bar
# already does when `<y>_sd` is absent.
fig, axes = plt.subplots(2, 4, figsize=(13, 7))
axes = axes.flatten()

SEROS = [('DP', 'HLA-DP'), ('DQ', 'HLA-DQ'), ('DR', 'HLA-DR'), ('H2', 'H2')]
for ax, (sero, title) in zip(axes[:4], SEROS):
    fx.draw_bar(ax, fx.rep('6_serotype', sero=sero, refs=True), title=title,
                ylim=(0.75, 1.0))
for ax, (sero, title) in zip(axes[4:], SEROS):
    fx.draw_bar(ax, fx.rep('7_serotype_ms', sero=sero, refs=True), title=title,
                ylim=(0.75, 1.0))

names = fx.rep('6_serotype', sero='DP', refs=True)
handles, labels = fx.legend_handles(names['full_name'], names['model'])
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.08),
           ncol=len(labels), frameon=False)
plt.tight_layout()

for row, lab in enumerate(['Qualitative', 'Mass Spectrometry']):
    pos = axes[row * 4].get_position()
    fig.text(pos.x0 - 0.05, (pos.y0 + pos.y1) / 2, lab,
             rotation=90, va='center', ha='center', fontsize=11)

plt.savefig('fig4.pdf', bbox_inches='tight')
plt.savefig('fig4.svg', bbox_inches='tight')
plt.show()


In [ ]:
# plot2b: the four distribution panels. The two published tools appear in
# Per Molecule only - see the caveat in the first cell.
fig, axes = plt.subplots(1, 4, figsize=(13, 4))
axes = axes.flatten()

# Panel a keeps the allele as the test set names it, an alpha/beta dimer, and does
# not collapse onto the beta chain. The collapse exists only so DeepNeo can be
# paired: it models no alpha chain and indexes its alleles by bare beta, so at
# this unit it shares no unit name with the others and its comparisons return
# n=0. It is plotted without an annotation for that reason, and its comparisons
# are reported on the collapsed unit in the supplementary instead. Collapsing
# would leave DeepNeo's own value unchanged and move no other method by more than
# 0.010 (Supplementary Table S15), so nothing is lost by reporting it there.
whole_alleles = fx.alleles('1_whole', refs=True)
lomo_beta = fx.by_beta(fx.lomo())
h2_beta = fx.by_beta(fx.alleles('5_h2'), value='roc_auc', unit='HLA_Name')

#: the methods that can be paired at the allele - everything but DeepNeo
PAIRABLE = [m for m in whole_alleles['full_name'].drop_duplicates() if m != 'DeepNeo']
#: brackets are drawn for pairs involving one of these
DRAWN = ['BLOSUM62', 'NetMHCIIpan-4.3', 'MixMHC2pred-2.0']

# All 15 pairs among PAIRABLE are tested and Holm-corrected, and every p value is
# in the supplementary. Ten of the twelve significant pairs are drawn: everything
# involving the BLOSUM62 baseline or one of the two published tools. The two left
# out are Chai-1 against ESMC 300M and against ESM3 Small, which the text
# describes and Supplementary Table S18 reports.
#
# Drawing all twelve puts the near-row labels on top of the boxes, and raising the
# axis until they fit would push a ROC-AUC plot past 1.3 and break the shared
# scale with panel (b). BLOSUM62 has to be in the drawn set even though it is the
# baseline rather than the subject: it is not distinguished from either tool
# (p_adj = 1.0 both), so with only the tool comparisons drawn it would appear
# unbracketed and read as differing from nothing, when it is in fact worse than
# all three PLM and structure representations.
#
# shared_units drops H2-IAg7, which NetMHCIIpan-4.3 has no model for, so the Holm
# family is one sample (n=57) rather than a mix of 57 and 58. DeepNeo keeps all of
# its own molecules: common_units passes through the methods it is not pairing.
fx.draw_box(axes[0], whole_alleles, unit='HLA_Name', y='roc_auc',
            title='Per Molecule', ylim=(0.5, 1.32), verbose=True,
            shared_units=PAIRABLE, stat_models=PAIRABLE, bracket_only=DRAWN)
fx.draw_box(axes[1], lomo_beta, unit='beta', y='roc_auc',
            title='LOMO - Whole', ylim=(0.5, 1.32), verbose=True)
fx.draw_box(axes[2], lomo_beta[lomo_beta['beta'].str.contains('H2')], unit='beta',
            y='roc_auc', title='LOMO - H2', ylim=(0.24, 0.84), verbose=True)
fx.draw_box(axes[3], h2_beta, unit='beta', y='roc_auc',
            title='H2-Out - H2', ylim=(0.24, 0.84), verbose=True)
# Preserve the original plotting ranges and annotation space, but do not label
# an impossible ROC-AUC value above 1.0.
for ax in axes[:2]:
    ax.set_yticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0])


for i, lab in enumerate(['a', 'b', 'c', 'd']):
    axes[i].text(-0.15, 1.15, lab, transform=axes[i].transAxes,
                 fontsize=12, fontweight='bold', va='top', ha='left')

# the legend carries all seven, because panel a draws all seven
names = whole_alleles[['model', 'full_name']].drop_duplicates()
handles, labels = fx.legend_handles(names['full_name'], names['model'])
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.13),
           ncol=len(labels), frameon=False)
plt.tight_layout()
plt.savefig('fig3.pdf', bbox_inches='tight')
plt.savefig('fig3.svg', bbox_inches='tight')
plt.show()
